# 04 — Volatility Features

This notebook comes after:

`03_WGI_Governance_Compilation.ipynb`

Purpose:

Create pre-shock volatility and stability features from standardized macroeconomic variables.

Main idea:

A country with more stable pre-shock macroeconomic conditions may have stronger structural resilience capacity.

This notebook creates candidate features for two analysis windows:

- 2007 shock structural window: `2000–2007`
- 2019 shock structural window: `2012–2019`

Candidate volatility variables:

- GDP growth volatility
- Inflation volatility
- Unemployment volatility
- Productivity growth / level volatility, if available and meaningful

Preferred first feature for the project:

`gdp_growth_stability`

Important:

- This notebook uses only **pre-shock years**.
- It does **not** use post-shock years.
- It does **not** use the recovery outcome.
- It does **not** decide final POSet variables.
- It does **not** impute missing values.
- It creates both volatility measures and direction-aligned stability measures where higher = better.

In [1]:
# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import re
import shutil
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

INPUT_FILE = (
    PROJECT_ROOT
    / "Data"
    / "Processed"
    / "00_Comparable_Raw_Files"
    / "Combined_Long"
    / "all_comparable_sources_long.csv"
)

RECOVERY_FILE = (
    PROJECT_ROOT
    / "Data"
    / "Processed"
    / "Recovery"
    / "country_recovery_summary_dynamic_baseline_2007_2019.csv"
)

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Volatility_Features"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "04_Volatility_Features"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

CLEAR_PREVIOUS_OUTPUTS = True

if CLEAR_PREVIOUS_OUTPUTS:
    for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR]:
        if folder.exists():
            shutil.rmtree(folder)

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Run ID:", RUN_ID)
print("Input file:", INPUT_FILE.resolve())
print("Recovery file:", RECOVERY_FILE.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())
print("Diagnostics folder:", DIAGNOSTICS_DIR.resolve())

Run ID: 20260622_033916
Input file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\00_Comparable_Raw_Files\Combined_Long\all_comparable_sources_long.csv
Recovery file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Recovery\country_recovery_summary_dynamic_baseline_2007_2019.csv
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Volatility_Features
Diagnostics folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\04_Volatility_Features


In [2]:
# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

PRE_SHOCK_WINDOWS = [
    {
        "shock_id": "2007",
        "shock_label": "shock_2007",
        "window_start": 2000,
        "window_end": 2007,
        "recovery_column": "Recovery_2007",
    },
    {
        "shock_id": "2019",
        "shock_label": "shock_2019",
        "window_start": 2012,
        "window_end": 2019,
        "recovery_column": "Recovery_2019",
    },
]

# Candidate variables for volatility/stability construction.
# The notebook will automatically skip missing variables.
VOLATILITY_VARIABLES = [
    {
        "variable": "gdp_growth",
        "feature_slug": "gdp_growth",
        "concept": "GDP growth volatility",
        "expected_unit": "percentage points",
        "preferred_feature": True,
        "interpretation": "Lower volatility is usually more stable; stability version is higher = better.",
    },
    {
        "variable": "inflation_cpi",
        "feature_slug": "inflation",
        "concept": "Inflation volatility",
        "expected_unit": "percentage points",
        "preferred_feature": False,
        "interpretation": "Lower volatility is usually more stable; stability version is higher = better.",
    },
    {
        "variable": "unemployment_rate",
        "feature_slug": "unemployment",
        "concept": "Unemployment volatility",
        "expected_unit": "percentage points",
        "preferred_feature": False,
        "interpretation": "Lower volatility is usually more stable; stability version is higher = better.",
    },
    {
        "variable": "productivity_gdp_per_hour",
        "feature_slug": "productivity_level",
        "concept": "Productivity level volatility",
        "expected_unit": "USD PPP per hour",
        "preferred_feature": False,
        "interpretation": "Interpret carefully; volatility of productivity level may reflect trend growth as well as instability.",
    },
]

MIN_YEARS_FOR_VOLATILITY = 4

print("Pre-shock windows:")
display(pd.DataFrame(PRE_SHOCK_WINDOWS))

print("Volatility variables:")
display(pd.DataFrame(VOLATILITY_VARIABLES))

Pre-shock windows:


,shock_id,shock_label,window_start,window_end,recovery_column
0,2007,shock_2007,2000,2007,Recovery_2007
1,2019,shock_2019,2012,2019,Recovery_2019


Volatility variables:


,variable,feature_slug,concept,expected_unit,preferred_feature,interpretation
0,gdp_growth,gdp_growth,GDP growth volatility,percentage points,True,Lower volatility is usually more stable; stabi...
1,inflation_cpi,inflation,Inflation volatility,percentage points,False,Lower volatility is usually more stable; stabi...
2,unemployment_rate,unemployment,Unemployment volatility,percentage points,False,Lower volatility is usually more stable; stabi...
3,productivity_gdp_per_hour,productivity_level,Productivity level volatility,USD PPP per hour,False,Interpret carefully; volatility of productivit...


In [3]:
# ------------------------------------------------------
# LOAD STANDARDIZED COMPARABLE LONG DATA
# ------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

df = pd.read_csv(INPUT_FILE)

required_columns = {
    "Country",
    "Year",
    "variable",
    "Value",
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Input dataset is missing required columns: {missing_columns}")

df["Country"] = df["Country"].astype(str).str.strip().str.upper()
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

df = df.dropna(subset=["Country", "Year", "variable", "Value"]).copy()
df["Year"] = df["Year"].astype(int)

if "country_name" not in df.columns:
    df["country_name"] = df["Country"]

df = df.sort_values(["variable", "Country", "Year"]).reset_index(drop=True)

available_variables = sorted(df["variable"].unique())

configured_variables = [item["variable"] for item in VOLATILITY_VARIABLES]
missing_configured_variables = sorted(set(configured_variables) - set(available_variables))

if missing_configured_variables:
    print("WARNING: These configured volatility variables are missing and will be skipped:")
    print(missing_configured_variables)

active_volatility_variables = [
    item
    for item in VOLATILITY_VARIABLES
    if item["variable"] in available_variables
]

if not active_volatility_variables:
    raise ValueError("None of the configured volatility variables are available in the input dataset.")

df_volatility_input = df[
    df["variable"].isin([item["variable"] for item in active_volatility_variables])
].copy()

df_volatility_input.to_csv(
    DIAGNOSTICS_DIR / "volatility_input_snapshot.csv",
    index=False,
)

print("Input rows:", len(df))
print("Volatility input rows:", len(df_volatility_input))
print("Countries:", df_volatility_input["Country"].nunique())
print("Variables used:", sorted(df_volatility_input["variable"].unique()))

display(df_volatility_input.head())

Input rows: 13446
Volatility input rows: 4629
Countries: 54
Variables used: ['gdp_growth', 'inflation_cpi', 'productivity_gdp_per_hour', 'unemployment_rate']


,Country,Country_Raw,country_name,Year,variable,Value,source_file,source_family,concept,expected_unit,raw_direction,is_derived,variable_notes,standardization_status,is_aggregate_region,Source_Used
3151,AUS,AUS,Australia,2000,gdp_growth,2.0315,OECD_GDP_Growth_2000_2025.csv,OECD_DBnomics,Real GDP growth,annual_percent_change,context_dependent,False,Used for recovery outcome and pre-shock GDP gr...,already_iso3,False,NaN
3152,AUS,AUS,Australia,2001,gdp_growth,3.9498,OECD_GDP_Growth_2000_2025.csv,OECD_DBnomics,Real GDP growth,annual_percent_change,context_dependent,False,Used for recovery outcome and pre-shock GDP gr...,already_iso3,False,NaN
3153,AUS,AUS,Australia,2002,gdp_growth,3.0913,OECD_GDP_Growth_2000_2025.csv,OECD_DBnomics,Real GDP growth,annual_percent_change,context_dependent,False,Used for recovery outcome and pre-shock GDP gr...,already_iso3,False,NaN
3154,AUS,AUS,Australia,2003,gdp_growth,4.2189,OECD_GDP_Growth_2000_2025.csv,OECD_DBnomics,Real GDP growth,annual_percent_change,context_dependent,False,Used for recovery outcome and pre-shock GDP gr...,already_iso3,False,NaN
3155,AUS,AUS,Australia,2004,gdp_growth,3.1555,OECD_GDP_Growth_2000_2025.csv,OECD_DBnomics,Real GDP growth,annual_percent_change,context_dependent,False,Used for recovery outcome and pre-shock GDP gr...,already_iso3,False,NaN


In [4]:
# ------------------------------------------------------
# LOAD RECOVERY OUTCOMES FOR DIAGNOSTIC ASSOCIATION ONLY
# ------------------------------------------------------
# Recovery is not used to construct volatility/stability features.
# It is only used later in this notebook for a diagnostic correlation table.

if RECOVERY_FILE.exists():
    recovery = pd.read_csv(RECOVERY_FILE)
    recovery["Country"] = recovery["Country"].astype(str).str.strip().str.upper()
    print("Recovery file loaded:", RECOVERY_FILE)
    print("Recovery rows:", len(recovery))
else:
    recovery = pd.DataFrame()
    print("Recovery file not found. Recovery diagnostic association will be skipped.")

display(recovery.head() if not recovery.empty else recovery)

Recovery file loaded: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Recovery\country_recovery_summary_dynamic_baseline_2007_2019.csv
Recovery rows: 44


,Country,country_name,Recovery_2007,Recovery_2019,Recovery_2007_recovery_status,Recovery_2007_shock_detection_status,Recovery_2007_first_negative_growth_year,Recovery_2007_baseline_year,Recovery_2007_baseline_available_in_growth_file,Recovery_2007_recovery_year,Recovery_2007_years_from_baseline_to_recovery,Recovery_2007_years_after_first_negative_to_recovery,Recovery_2007_last_index_year,Recovery_2007_last_index_value,Recovery_2019_recovery_status,Recovery_2019_shock_detection_status,Recovery_2019_first_negative_growth_year,Recovery_2019_baseline_year,Recovery_2019_baseline_available_in_growth_file,Recovery_2019_recovery_year,Recovery_2019_years_from_baseline_to_recovery,Recovery_2019_years_after_first_negative_to_recovery,Recovery_2019_last_index_year,Recovery_2019_last_index_value
0,AUS,Australia,0.0000,0.0000,no_negative_growth_in_shock_window,no_negative_growth_in_shock_window,NaN,NaN,False,NaN,0.0000,0.0000,NaN,NaN,no_negative_growth_in_shock_window,no_negative_growth_in_shock_window,NaN,NaN,False,NaN,0.0000,0.0000,NaN,NaN
1,AUT,Austria,2.0000,2.0000,recovered,negative_growth_found,2009.0000,2008.0000,True,2011.0000,3.0000,2.0000,2011.0000,101.0314,recovered,negative_growth_found,2020.0000,2019.0000,True,2022.0000,3.0000,2.0000,2022.0000,103.5338
2,BEL,Belgium,1.0000,1.0000,recovered,negative_growth_found,2009.0000,2008.0000,True,2010.0000,2.0000,1.0000,2010.0000,100.7522,recovered,negative_growth_found,2020.0000,2019.0000,True,2021.0000,2.0000,1.0000,2021.0000,101.1617
3,BRA,Brazil,1.0000,1.0000,recovered,negative_growth_found,2009.0000,2008.0000,True,2010.0000,2.0000,1.0000,2010.0000,107.3929,recovered,negative_growth_found,2020.0000,2019.0000,True,2021.0000,2.0000,1.0000,2021.0000,101.3298
4,CAN,Canada,1.0000,1.0000,recovered,negative_growth_found,2009.0000,2008.0000,True,2010.0000,2.0000,1.0000,2010.0000,100.0856,recovered,negative_growth_found,2020.0000,2019.0000,True,2021.0000,2.0000,1.0000,2021.0000,100.6125


In [5]:
# ------------------------------------------------------
# RAW COVERAGE BY VARIABLE AND WINDOW
# ------------------------------------------------------

coverage_rows = []

for window in PRE_SHOCK_WINDOWS:
    shock_id = window["shock_id"]
    shock_label = window["shock_label"]
    start = window["window_start"]
    end = window["window_end"]
    expected_years = list(range(start, end + 1))

    for item in active_volatility_variables:
        variable = item["variable"]
        feature_slug = item["feature_slug"]

        subset = df_volatility_input[
            (df_volatility_input["variable"] == variable)
            & (df_volatility_input["Year"] >= start)
            & (df_volatility_input["Year"] <= end)
        ].copy()

        country_summary = (
            subset
            .groupby("Country")
            .agg(
                years_available=("Year", "nunique"),
                min_year=("Year", "min"),
                max_year=("Year", "max"),
                min_value=("Value", "min"),
                median_value=("Value", "median"),
                max_value=("Value", "max"),
            )
            .reset_index()
        )

        coverage_rows.append({
            "shock_id": shock_id,
            "shock_label": shock_label,
            "window_start": start,
            "window_end": end,
            "variable": variable,
            "feature_slug": feature_slug,
            "expected_year_count": len(expected_years),
            "countries_with_any_data": subset["Country"].nunique(),
            "countries_with_min_years": int((country_summary["years_available"] >= MIN_YEARS_FOR_VOLATILITY).sum()),
            "median_years_available": country_summary["years_available"].median() if not country_summary.empty else np.nan,
            "max_years_available": country_summary["years_available"].max() if not country_summary.empty else np.nan,
        })

volatility_window_coverage_summary = pd.DataFrame(coverage_rows)

volatility_window_coverage_summary.to_csv(
    DIAGNOSTICS_DIR / "volatility_window_coverage_summary.csv",
    index=False,
)

display(volatility_window_coverage_summary)

,shock_id,shock_label,window_start,window_end,variable,feature_slug,expected_year_count,countries_with_any_data,countries_with_min_years,median_years_available,max_years_available
0,2007,shock_2007,2000,2007,gdp_growth,gdp_growth,8,44,44,8.0000,8
1,2007,shock_2007,2000,2007,inflation_cpi,inflation,8,47,47,8.0000,8
2,2007,shock_2007,2000,2007,unemployment_rate,unemployment,8,51,48,8.0000,8
3,2007,shock_2007,2000,2007,productivity_gdp_per_hour,productivity_level,8,40,40,8.0000,8
4,2019,shock_2019,2012,2019,gdp_growth,gdp_growth,8,44,44,8.0000,8
5,2019,shock_2019,2012,2019,inflation_cpi,inflation,8,48,47,8.0000,8
6,2019,shock_2019,2012,2019,unemployment_rate,unemployment,8,52,51,8.0000,8
7,2019,shock_2019,2012,2019,productivity_gdp_per_hour,productivity_level,8,41,41,8.0000,8


In [6]:
# ------------------------------------------------------
# VOLATILITY FEATURE CONSTRUCTION HELPERS
# ------------------------------------------------------

def min_max_scale_higher_better_from_lower_better(series):
    values = pd.to_numeric(series, errors="coerce")
    min_value = values.min(skipna=True)
    max_value = values.max(skipna=True)

    if pd.isna(min_value) or pd.isna(max_value):
        return pd.Series(np.nan, index=series.index)

    if np.isclose(max_value, min_value):
        return pd.Series(100.0, index=series.index)

    return 100.0 * (max_value - values) / (max_value - min_value)


def min_max_scale_higher_better(series):
    values = pd.to_numeric(series, errors="coerce")
    min_value = values.min(skipna=True)
    max_value = values.max(skipna=True)

    if pd.isna(min_value) or pd.isna(max_value):
        return pd.Series(np.nan, index=series.index)

    if np.isclose(max_value, min_value):
        return pd.Series(100.0, index=series.index)

    return 100.0 * (values - min_value) / (max_value - min_value)


def calculate_country_window_stats(subset, expected_years):
    expected_year_count = len(expected_years)

    stats = (
        subset
        .groupby(["Country", "country_name", "variable"])
        .agg(
            years_available=("Year", "nunique"),
            min_year=("Year", "min"),
            max_year=("Year", "max"),
            mean_value=("Value", "mean"),
            median_value=("Value", "median"),
            min_value=("Value", "min"),
            max_value=("Value", "max"),
            volatility_std=("Value", lambda x: x.std(ddof=1)),
            volatility_mad=("Value", lambda x: np.mean(np.abs(x - np.mean(x))) if len(x) > 0 else np.nan),
            first_value=("Value", "first"),
            last_value=("Value", "last"),
        )
        .reset_index()
    )

    stats["expected_year_count"] = expected_year_count
    stats["coverage_rate_in_window"] = stats["years_available"] / expected_year_count
    stats["sufficient_years_for_volatility"] = stats["years_available"] >= MIN_YEARS_FOR_VOLATILITY

    stats.loc[~stats["sufficient_years_for_volatility"], "volatility_std"] = np.nan
    stats.loc[~stats["sufficient_years_for_volatility"], "volatility_mad"] = np.nan

    stats["range_value"] = stats["max_value"] - stats["min_value"]
    stats["change_first_to_last"] = stats["last_value"] - stats["first_value"]

    return stats

In [7]:
# ------------------------------------------------------
# BUILD VOLATILITY FEATURES BY COUNTRY-SHOCK-VARIABLE
# ------------------------------------------------------

feature_rows = []

for window in PRE_SHOCK_WINDOWS:
    shock_id = window["shock_id"]
    shock_label = window["shock_label"]
    start = window["window_start"]
    end = window["window_end"]
    expected_years = list(range(start, end + 1))

    for item in active_volatility_variables:
        variable = item["variable"]
        feature_slug = item["feature_slug"]

        subset = df_volatility_input[
            (df_volatility_input["variable"] == variable)
            & (df_volatility_input["Year"] >= start)
            & (df_volatility_input["Year"] <= end)
        ].copy()

        subset = subset.sort_values(["Country", "Year"]).copy()

        if subset.empty:
            continue

        stats = calculate_country_window_stats(subset, expected_years)

        stats["shock_id"] = shock_id
        stats["shock_label"] = shock_label
        stats["window_start"] = start
        stats["window_end"] = end
        stats["feature_slug"] = feature_slug
        stats["concept"] = item["concept"]
        stats["expected_unit"] = item["expected_unit"]
        stats["preferred_feature"] = item["preferred_feature"]
        stats["interpretation"] = item["interpretation"]

        feature_rows.append(stats)

volatility_features_long = pd.concat(feature_rows, ignore_index=True)

# Direction-aligned stability measures.
# Lower volatility -> higher stability.
volatility_features_long["stability_negative_std"] = -volatility_features_long["volatility_std"]
volatility_features_long["stability_negative_mad"] = -volatility_features_long["volatility_mad"]

# 0-100 stability scores are scaled within each shock_id + variable.
# These are useful for diagnostics/visuals, not raw source measures.
volatility_features_long["stability_std_0_100"] = np.nan
volatility_features_long["stability_mad_0_100"] = np.nan

for (shock_id, variable), idx in volatility_features_long.groupby(["shock_id", "variable"]).groups.items():
    idx = list(idx)

    volatility_features_long.loc[idx, "stability_std_0_100"] = (
        min_max_scale_higher_better_from_lower_better(
            volatility_features_long.loc[idx, "volatility_std"]
        )
    )

    volatility_features_long.loc[idx, "stability_mad_0_100"] = (
        min_max_scale_higher_better_from_lower_better(
            volatility_features_long.loc[idx, "volatility_mad"]
        )
    )

volatility_features_long = volatility_features_long[
    [
        "Country",
        "country_name",
        "shock_id",
        "shock_label",
        "window_start",
        "window_end",
        "variable",
        "feature_slug",
        "concept",
        "expected_unit",
        "preferred_feature",
        "years_available",
        "expected_year_count",
        "coverage_rate_in_window",
        "sufficient_years_for_volatility",
        "mean_value",
        "median_value",
        "min_value",
        "max_value",
        "range_value",
        "first_value",
        "last_value",
        "change_first_to_last",
        "volatility_std",
        "volatility_mad",
        "stability_negative_std",
        "stability_negative_mad",
        "stability_std_0_100",
        "stability_mad_0_100",
        "interpretation",
    ]
].sort_values(["shock_id", "variable", "Country"]).reset_index(drop=True)

volatility_features_long.to_csv(
    PROCESSED_DIR / "volatility_features_country_shock_variable_long.csv",
    index=False,
)

volatility_features_long.to_csv(
    DIAGNOSTICS_DIR / "volatility_features_country_shock_variable_long.csv",
    index=False,
)

print("Volatility feature rows:", len(volatility_features_long))
display(volatility_features_long.head(80))

Volatility feature rows: 367


,Country,country_name,shock_id,shock_label,window_start,window_end,variable,feature_slug,concept,expected_unit,preferred_feature,years_available,expected_year_count,coverage_rate_in_window,sufficient_years_for_volatility,mean_value,median_value,min_value,max_value,range_value,first_value,last_value,change_first_to_last,volatility_std,volatility_mad,stability_negative_std,stability_negative_mad,stability_std_0_100,stability_mad_0_100,interpretation
0,AUS,Australia,2007,shock_2007,2000,2007,gdp_growth,gdp_growth,GDP growth volatility,percentage points,True,8,8,1.0000,True,3.3289,3.3897,2.0315,4.2189,2.1875,2.0315,3.6240,1.5925,0.7130,0.5666,-0.7130,-0.5666,100.0000,100.0000,Lower volatility is usually more stable; stabi...
1,AUT,Austria,2007,shock_2007,2000,2007,gdp_growth,gdp_growth,GDP growth volatility,percentage points,True,8,8,1.0000,True,2.3828,2.4428,1.1416,3.7752,2.6337,3.1895,3.7752,0.5857,0.9926,0.8170,-0.9926,-0.8170,93.0853,89.2635,Lower volatility is usually more stable; stabi...
2,BEL,Belgium,2007,shock_2007,2000,2007,gdp_growth,gdp_growth,GDP growth volatility,percentage points,True,8,8,1.0000,True,2.4604,2.4370,1.0380,3.7167,2.6787,3.7167,3.6769,-0.0398,1.1188,0.9189,-1.1188,-0.9189,89.9631,84.8953,Lower volatility is usually more stable; stabi...
3,BRA,Brazil,2007,shock_2007,2000,2007,gdp_growth,gdp_growth,GDP growth volatility,percentage points,True,8,8,1.0000,True,3.6208,3.5821,1.1408,6.0699,4.9290,4.3879,6.0699,1.6819,1.8088,1.4242,-1.8088,-1.4242,72.9009,63.2269,Lower volatility is usually more stable; stabi...
4,CAN,Canada,2007,shock_2007,2000,2007,gdp_growth,gdp_growth,GDP growth volatility,percentage points,True,8,8,1.0000,True,2.8512,2.8186,1.8064,5.1385,3.3322,5.1385,2.0499,-3.0886,1.0805,0.7589,-1.0805,-0.7589,90.9102,91.7548,Lower volatility is usually more stable; stabi...
5,CHE,Switzerland,2007,shock_2007,2000,2007,gdp_growth,gdp_growth,GDP growth volatility,percentage points,True,8,8,1.0000,True,2.3723,2.7695,-0.2842,4.1343,4.4185,3.6332,3.9832,0.3500,1.6607,1.3571,-1.6607,-1.3571,76.5645,66.1033,Lower volatility is usually more stable; stabi...
6,CHL,Chile,2007,shock_2007,2000,2007,gdp_growth,gdp_growth,GDP growth volatility,percentage points,True,8,8,1.0000,True,4.9727,5.0699,3.1540,6.6743,3.5202,4.9716,5.1682,0.1966,1.2730,0.9597,-1.2730,-0.9597,86.1515,83.1434,Lower volatility is usually more stable; stabi...
7,CHN,China,2007,shock_2007,2000,2007,gdp_growth,gdp_growth,GDP growth volatility,percentage points,True,8,8,1.0000,True,10.5572,10.0758,8.3358,14.2308,5.8951,8.4900,14.2308,5.7409,2.0901,1.6687,-2.0901,-1.6687,65.9457,52.7410,Lower volatility is usually more stable; stabi...
8,COL,Colombia,2007,shock_2007,2000,2007,gdp_growth,gdp_growth,GDP growth volatility,percentage points,True,8,8,1.0000,True,4.2705,4.3125,1.6779,6.7382,5.0603,2.5692,6.7382,4.1690,1.9366,1.6031,-1.9366,-1.6031,69.7407,55.5532,Lower volatility is usually more stable; stabi...
9,CRI,Costa Rica,2007,shock_2007,2000,2007,gdp_growth,gdp_growth,GDP growth volatility,percentage points,True,8,8,1.0000,True,4.8796,4.1469,3.4169,8.2151,4.7983,3.8687,8.2151,4.3464,1.8339,1.4456,-1.8339,-1.4456,72.2816,62.3067,Lower volatility is usually more stable; stabi...


In [8]:
# ------------------------------------------------------
# CREATE WIDE FEATURE TABLES
# ------------------------------------------------------

feature_value_columns = [
    "volatility_std",
    "volatility_mad",
    "stability_negative_std",
    "stability_negative_mad",
    "stability_std_0_100",
    "stability_mad_0_100",
    "mean_value",
    "median_value",
    "change_first_to_last",
    "years_available",
    "coverage_rate_in_window",
]

wide_frames = []

base_keys = ["Country", "country_name", "shock_id", "shock_label", "window_start", "window_end"]

for value_col in feature_value_columns:
    temp = volatility_features_long.copy()

    temp["wide_feature_name"] = (
        temp["feature_slug"]
        + "_"
        + value_col
        + "_pre_"
        + temp["shock_id"].astype(str)
    )

    wide_temp = (
        temp
        .pivot_table(
            index=base_keys,
            columns="wide_feature_name",
            values=value_col,
            aggfunc="first",
        )
        .reset_index()
    )

    wide_frames.append(wide_temp)

volatility_features_wide = wide_frames[0]

for frame in wide_frames[1:]:
    volatility_features_wide = volatility_features_wide.merge(
        frame,
        on=base_keys,
        how="outer",
    )

volatility_features_wide = volatility_features_wide.sort_values(["shock_id", "Country"]).reset_index(drop=True)

volatility_features_wide.to_csv(
    PROCESSED_DIR / "volatility_features_country_shock_wide.csv",
    index=False,
)

volatility_features_wide.to_csv(
    DIAGNOSTICS_DIR / "volatility_features_country_shock_wide.csv",
    index=False,
)

# Also create one-row-per-country wide table across both shocks.
country_level_keys = ["Country", "country_name"]

cross_shock_frames = []

for value_col in feature_value_columns:
    temp = volatility_features_long.copy()

    temp["wide_feature_name"] = (
        temp["feature_slug"]
        + "_"
        + value_col
        + "_pre_"
        + temp["shock_id"].astype(str)
    )

    cross_temp = (
        temp
        .pivot_table(
            index=country_level_keys,
            columns="wide_feature_name",
            values=value_col,
            aggfunc="first",
        )
        .reset_index()
    )

    cross_shock_frames.append(cross_temp)

volatility_features_country_wide = cross_shock_frames[0]

for frame in cross_shock_frames[1:]:
    volatility_features_country_wide = volatility_features_country_wide.merge(
        frame,
        on=country_level_keys,
        how="outer",
    )

volatility_features_country_wide = volatility_features_country_wide.sort_values("Country").reset_index(drop=True)

volatility_features_country_wide.to_csv(
    PROCESSED_DIR / "volatility_features_country_wide.csv",
    index=False,
)

print("Country-shock wide rows:", len(volatility_features_wide))
print("Country-wide rows:", len(volatility_features_country_wide))

display(volatility_features_wide.head())
display(volatility_features_country_wide.head())

Country-shock wide rows: 106
Country-wide rows: 54


wide_feature_name,Country,country_name,shock_id,shock_label,window_start,window_end,gdp_growth_volatility_std_pre_2007,gdp_growth_volatility_std_pre_2019,inflation_volatility_std_pre_2007,inflation_volatility_std_pre_2019,productivity_level_volatility_std_pre_2007,productivity_level_volatility_std_pre_2019,unemployment_volatility_std_pre_2007,unemployment_volatility_std_pre_2019,gdp_growth_volatility_mad_pre_2007,gdp_growth_volatility_mad_pre_2019,inflation_volatility_mad_pre_2007,inflation_volatility_mad_pre_2019,productivity_level_volatility_mad_pre_2007,productivity_level_volatility_mad_pre_2019,unemployment_volatility_mad_pre_2007,unemployment_volatility_mad_pre_2019,gdp_growth_stability_negative_std_pre_2007,gdp_growth_stability_negative_std_pre_2019,inflation_stability_negative_std_pre_2007,inflation_stability_negative_std_pre_2019,productivity_level_stability_negative_std_pre_2007,productivity_level_stability_negative_std_pre_2019,unemployment_stability_negative_std_pre_2007,unemployment_stability_negative_std_pre_2019,gdp_growth_stability_negative_mad_pre_2007,gdp_growth_stability_negative_mad_pre_2019,inflation_stability_negative_mad_pre_2007,inflation_stability_negative_mad_pre_2019,productivity_level_stability_negative_mad_pre_2007,productivity_level_stability_negative_mad_pre_2019,unemployment_stability_negative_mad_pre_2007,unemployment_stability_negative_mad_pre_2019,gdp_growth_stability_std_0_100_pre_2007,gdp_growth_stability_std_0_100_pre_2019,inflation_stability_std_0_100_pre_2007,inflation_stability_std_0_100_pre_2019,productivity_level_stability_std_0_100_pre_2007,productivity_level_stability_std_0_100_pre_2019,unemployment_stability_std_0_100_pre_2007,unemployment_stability_std_0_100_pre_2019,gdp_growth_stability_mad_0_100_pre_2007,gdp_growth_stability_mad_0_100_pre_2019,inflation_stability_mad_0_100_pre_2007,inflation_stability_mad_0_100_pre_2019,productivity_level_stability_mad_0_100_pre_2007,productivity_level_stability_mad_0_100_pre_2019,unemployment_stability_mad_0_100_pre_2007,unemployment_stability_mad_0_100_pre_2019,gdp_growth_mean_value_pre_2007,gdp_growth_mean_value_pre_2019,inflation_mean_value_pre_2007,inflation_mean_value_pre_2019,productivity_level_mean_value_pre_2007,productivity_level_mean_value_pre_2019,unemployment_mean_value_pre_2007,unemployment_mean_value_pre_2019,gdp_growth_median_value_pre_2007,gdp_growth_median_value_pre_2019,inflation_median_value_pre_2007,inflation_median_value_pre_2019,productivity_level_median_value_pre_2007,productivity_level_median_value_pre_2019,unemployment_median_value_pre_2007,unemployment_median_value_pre_2019,gdp_growth_change_first_to_last_pre_2007,gdp_growth_change_first_to_last_pre_2019,inflation_change_first_to_last_pre_2007,inflation_change_first_to_last_pre_2019,productivity_level_change_first_to_last_pre_2007,productivity_level_change_first_to_last_pre_2019,unemployment_change_first_to_last_pre_2007,unemployment_change_first_to_last_pre_2019,gdp_growth_years_available_pre_2007,gdp_growth_years_available_pre_2019,inflation_years_available_pre_2007,inflation_years_available_pre_2019,productivity_level_years_available_pre_2007,productivity_level_years_available_pre_2019,unemployment_years_available_pre_2007,unemployment_years_available_pre_2019,gdp_growth_coverage_rate_in_window_pre_2007,gdp_growth_coverage_rate_in_window_pre_2019,inflation_coverage_rate_in_window_pre_2007,inflation_coverage_rate_in_window_pre_2019,productivity_level_coverage_rate_in_window_pre_2007,productivity_level_coverage_rate_in_window_pre_2019,unemployment_coverage_rate_in_window_pre_2007,unemployment_coverage_rate_in_window_pre_2019
0,AUS,Australia,2007,shock_2007,2000,2007,0.7130,NaN,0.8605,NaN,1.8835,NaN,0.8455,NaN,0.5666,NaN,0.7146,NaN,1.5233,NaN,0.7175,NaN,-0.7130,NaN,-0.8605,NaN,-1.8835,NaN,-0.8455,NaN,-0.5666,NaN,-0.7146,NaN,-1.5233,NaN,-0.7175,NaN,100.0000,NaN,97.0362,NaN,69.5619,NaN,86.6022,NaN,100.0000,NaN,97.1921,NaN,70.7432,NaN,85.2081,NaN,3.3289,NaN,3.1871,NaN,57.4

wide_feature_name,Country,country_name,gdp_growth_volatility_std_pre_2007,gdp_growth_volatility_std_pre_2019,inflation_volatility_std_pre_2007,inflation_volatility_std_pre_2019,productivity_level_volatility_std_pre_2007,productivity_level_volatility_std_pre_2019,unemployment_volatility_std_pre_2007,unemployment_volatility_std_pre_2019,gdp_growth_volatility_mad_pre_2007,gdp_growth_volatility_mad_pre_2019,inflation_volatility_mad_pre_2007,inflation_volatility_mad_pre_2019,productivity_level_volatility_mad_pre_2007,productivity_level_volatility_mad_pre_2019,unemployment_volatility_mad_pre_2007,unemployment_volatility_mad_pre_2019,gdp_growth_stability_negative_std_pre_2007,gdp_growth_stability_negative_std_pre_2019,inflation_stability_negative_std_pre_2007,inflation_stability_negative_std_pre_2019,productivity_level_stability_negative_std_pre_2007,productivity_level_stability_negative_std_pre_2019,unemployment_stability_negative_std_pre_2007,unemployment_stability_negative_std_pre_2019,gdp_growth_stability_negative_mad_pre_2007,gdp_growth_stability_negative_mad_pre_2019,inflation_stability_negative_mad_pre_2007,inflation_stability_negative_mad_pre_2019,productivity_level_stability_negative_mad_pre_2007,productivity_level_stability_negative_mad_pre_2019,unemployment_stability_negative_mad_pre_2007,unemployment_stability_negative_mad_pre_2019,gdp_growth_stability_std_0_100_pre_2007,gdp_growth_stability_std_0_100_pre_2019,inflation_stability_std_0_100_pre_2007,inflation_stability_std_0_100_pre_2019,productivity_level_stability_std_0_100_pre_2007,productivity_level_stability_std_0_100_pre_2019,unemployment_stability_std_0_100_pre_2007,unemployment_stability_std_0_100_pre_2019,gdp_growth_stability_mad_0_100_pre_2007,gdp_growth_stability_mad_0_100_pre_2019,inflation_stability_mad_0_100_pre_2007,inflation_stability_mad_0_100_pre_2019,productivity_level_stability_mad_0_100_pre_2007,productivity_level_stability_mad_0_100_pre_2019,unemployment_stability_mad_0_100_pre_2007,unemployment_stability_mad_0_100_pre_2019,gdp_growth_mean_value_pre_2007,gdp_growth_mean_value_pre_2019,inflation_mean_value_pre_2007,inflation_mean_value_pre_2019,productivity_level_mean_value_pre_2007,productivity_level_mean_value_pre_2019,unemployment_mean_value_pre_2007,unemployment_mean_value_pre_2019,gdp_growth_median_value_pre_2007,gdp_growth_median_value_pre_2019,inflation_median_value_pre_2007,inflation_median_value_pre_2019,productivity_level_median_value_pre_2007,productivity_level_median_value_pre_2019,unemployment_median_value_pre_2007,unemployment_median_value_pre_2019,gdp_growth_change_first_to_last_pre_2007,gdp_growth_change_first_to_last_pre_2019,inflation_change_first_to_last_pre_2007,inflation_change_first_to_last_pre_2019,productivity_level_change_first_to_last_pre_2007,productivity_level_change_first_to_last_pre_2019,unemployment_change_first_to_last_pre_2007,unemployment_change_first_to_last_pre_2019,gdp_growth_years_available_pre_2007,gdp_growth_years_available_pre_2019,inflation_years_available_pre_2007,inflation_years_available_pre_2019,productivity_level_years_available_pre_2007,productivity_level_years_available_pre_2019,unemployment_years_available_pre_2007,unemployment_years_available_pre_2019,gdp_growth_coverage_rate_in_window_pre_2007,gdp_growth_coverage_rate_in_window_pre_2019,inflation_coverage_rate_in_window_pre_2007,inflation_coverage_rate_in_window_pre_2019,productivity_level_coverage_rate_in_window_pre_2007,productivity_level_coverage_rate_in_window_pre_2019,unemployment_coverage_rate_in_window_pre_2007,unemployment_coverage_rate_in_window_pre_2019
0,ARG,Argentina,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.9555,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.8142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.9555,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.8142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,82.4398,NaN,NaN,NaN,NaN,NaN,NaN,NaN,81.6889,NaN,NaN,NaN,43.9128,NaN,NaN,NaN,8.2018,NaN,NaN,NaN,43.9128,NaN,NaN,NaN,8.2305,NaN,NaN,NaN,19.2711,NaN,NaN,NaN,1.3390,NaN,NaN,NaN,2.0000,NaN,NaN,NaN,4.0000,NaN,NaN,Na

In [9]:
# ------------------------------------------------------
# PREFERRED GDP GROWTH STABILITY OUTPUT
# ------------------------------------------------------
# This compact file is likely the most useful output from this notebook.

preferred_gdp_growth_stability = volatility_features_long[
    volatility_features_long["variable"] == "gdp_growth"
].copy()

preferred_gdp_growth_stability = preferred_gdp_growth_stability[
    [
        "Country",
        "country_name",
        "shock_id",
        "shock_label",
        "window_start",
        "window_end",
        "years_available",
        "expected_year_count",
        "coverage_rate_in_window",
        "sufficient_years_for_volatility",
        "mean_value",
        "volatility_std",
        "stability_negative_std",
        "stability_std_0_100",
        "volatility_mad",
        "stability_negative_mad",
        "stability_mad_0_100",
    ]
].sort_values(["shock_id", "Country"]).reset_index(drop=True)

preferred_gdp_growth_stability.to_csv(
    PROCESSED_DIR / "gdp_growth_stability_features_country_shock.csv",
    index=False,
)

preferred_gdp_growth_stability.to_csv(
    DIAGNOSTICS_DIR / "gdp_growth_stability_features_country_shock.csv",
    index=False,
)

display(preferred_gdp_growth_stability.head(80))

,Country,country_name,shock_id,shock_label,window_start,window_end,years_available,expected_year_count,coverage_rate_in_window,sufficient_years_for_volatility,mean_value,volatility_std,stability_negative_std,stability_std_0_100,volatility_mad,stability_negative_mad,stability_mad_0_100
0,AUS,Australia,2007,shock_2007,2000,2007,8,8,1.0000,True,3.3289,0.7130,-0.7130,100.0000,0.5666,-0.5666,100.0000
1,AUT,Austria,2007,shock_2007,2000,2007,8,8,1.0000,True,2.3828,0.9926,-0.9926,93.0853,0.8170,-0.8170,89.2635
2,BEL,Belgium,2007,shock_2007,2000,2007,8,8,1.0000,True,2.4604,1.1188,-1.1188,89.9631,0.9189,-0.9189,84.8953
3,BRA,Brazil,2007,shock_2007,2000,2007,8,8,1.0000,True,3.6208,1.8088,-1.8088,72.9009,1.4242,-1.4242,63.2269
4,CAN,Canada,2007,shock_2007,2000,2007,8,8,1.0000,True,2.8512,1.0805,-1.0805,90.9102,0.7589,-0.7589,91.7548
5,CHE,Switzerland,2007,shock_2007,2000,2007,8,8,1.0000,True,2.3723,1.6607,-1.6607,76.5645,1.3571,-1.3571,66.1033
6,CHL,Chile,2007,shock_2007,2000,2007,8,8,1.0000,True,4.9727,1.2730,-1.2730,86.1515,0.9597,-0.9597,83.1434
7,CHN,China,2007,shock_2007,2000,2007,8,8,1.0000,True,10.5572,2.0901,-2.0901,65.9457,1.6687,-1.6687,52.7410
8,COL,Colombia,2007,shock_2007,2000,2007,8,8,1.0000,True,4.2705,1.9366,-1.9366,69.7407,1.6031,-1.6031,55.5532
9,CRI,Costa Rica,2007,shock_2007,2000,2007,8,8,1.0000,True,4.8796,1.8339,-1.8339,72.2816,1.4456,-1.4456,62.3067


In [10]:
# ------------------------------------------------------
# DIAGNOSTIC ASSOCIATION WITH RECOVERY OUTCOMES
# ------------------------------------------------------
# Recovery is not used to construct volatility/stability features.
# This is only a diagnostic check for later interpretation.

if recovery.empty:
    volatility_recovery_diagnostic = pd.DataFrame({
        "message": ["Recovery file not available, diagnostic association skipped."]
    })
else:
    diagnostic_rows = []

    for window in PRE_SHOCK_WINDOWS:
        shock_id = window["shock_id"]
        recovery_column = window["recovery_column"]

        if recovery_column not in recovery.columns:
            continue

        subset = volatility_features_long[
            volatility_features_long["shock_id"] == shock_id
        ].copy()

        subset = subset.merge(
            recovery[["Country", recovery_column]],
            on="Country",
            how="left",
        )

        subset = subset.rename(columns={recovery_column: "recovery_value"})

        for variable in sorted(subset["variable"].dropna().unique()):
            var_subset = subset[
                (subset["variable"] == variable)
                & subset["volatility_std"].notna()
                & subset["stability_std_0_100"].notna()
                & subset["recovery_value"].notna()
            ].copy()

            if len(var_subset) >= 5:
                corr_volatility = var_subset["volatility_std"].corr(var_subset["recovery_value"])
                corr_stability = var_subset["stability_std_0_100"].corr(var_subset["recovery_value"])
            else:
                corr_volatility = np.nan
                corr_stability = np.nan

            diagnostic_rows.append({
                "shock_id": shock_id,
                "recovery_column": recovery_column,
                "variable": variable,
                "countries_used": len(var_subset),
                "correlation_volatility_std_with_recovery": corr_volatility,
                "correlation_stability_std_0_100_with_recovery": corr_stability,
                "note": "Diagnostic only. Recovery is not used to construct features and this is not causal evidence.",
            })

    volatility_recovery_diagnostic = pd.DataFrame(diagnostic_rows)

volatility_recovery_diagnostic.to_csv(
    DIAGNOSTICS_DIR / "volatility_recovery_diagnostic_correlations.csv",
    index=False,
)

display(volatility_recovery_diagnostic)

,shock_id,recovery_column,variable,countries_used,correlation_volatility_std_with_recovery,correlation_stability_std_0_100_with_recovery,note
0,2007,Recovery_2007,gdp_growth,44,-0.1131,0.1131,Diagnostic only. Recovery is not used to const...
1,2007,Recovery_2007,inflation_cpi,44,-0.1592,0.1592,Diagnostic only. Recovery is not used to const...
2,2007,Recovery_2007,productivity_gdp_per_hour,37,0.1045,-0.1045,Diagnostic only. Recovery is not used to const...
3,2007,Recovery_2007,unemployment_rate,42,0.0943,-0.0943,Diagnostic only. Recovery is not used to const...
4,2019,Recovery_2019,gdp_growth,43,-0.1265,0.1265,Diagnostic only. Recovery is not used to const...
5,2019,Recovery_2019,inflation_cpi,43,-0.1728,0.1728,Diagnostic only. Recovery is not used to const...
6,2019,Recovery_2019,productivity_gdp_per_hour,38,-0.4282,0.4282,Diagnostic only. Recovery is not used to const...
7,2019,Recovery_2019,unemployment_rate,41,0.0464,-0.0464,Diagnostic only. Recovery is not used to const...


In [11]:
# ------------------------------------------------------
# FEATURE COVERAGE AND QUALITY FLAGS
# ------------------------------------------------------

feature_quality_summary = (
    volatility_features_long
    .groupby(["shock_id", "variable", "feature_slug", "concept"])
    .agg(
        countries_with_any_data=("Country", "nunique"),
        countries_with_sufficient_years=("sufficient_years_for_volatility", "sum"),
        median_years_available=("years_available", "median"),
        min_years_available=("years_available", "min"),
        max_years_available=("years_available", "max"),
        median_volatility_std=("volatility_std", "median"),
        min_volatility_std=("volatility_std", "min"),
        max_volatility_std=("volatility_std", "max"),
    )
    .reset_index()
)

feature_quality_summary["sufficient_years_country_rate"] = (
    feature_quality_summary["countries_with_sufficient_years"]
    / feature_quality_summary["countries_with_any_data"]
)

def classify_feature_quality(row):
    if row["countries_with_sufficient_years"] >= 35 and row["sufficient_years_country_rate"] >= 0.80:
        return "strong_candidate"

    if row["countries_with_sufficient_years"] >= 25 and row["sufficient_years_country_rate"] >= 0.60:
        return "review_candidate"

    return "weak_coverage_review"

feature_quality_summary["feature_quality_recommendation"] = feature_quality_summary.apply(
    classify_feature_quality,
    axis=1,
)

feature_quality_summary["note"] = "Diagnostic only; final variable selection happens during Pre-POSet EDA."

feature_quality_summary.to_csv(
    DIAGNOSTICS_DIR / "volatility_feature_quality_summary.csv",
    index=False,
)

display(feature_quality_summary)

,shock_id,variable,feature_slug,concept,countries_with_any_data,countries_with_sufficient_years,median_years_available,min_years_available,max_years_available,median_volatility_std,min_volatility_std,max_volatility_std,sufficient_years_country_rate,feature_quality_recommendation,note
0,2007,gdp_growth,gdp_growth,GDP growth volatility,44,44,8.0000,8,8,1.5573,0.7130,4.7569,1.0000,strong_candidate,Diagnostic only; final variable selection happ...
1,2007,inflation_cpi,inflation,Inflation volatility,47,47,8.0000,8,8,1.0300,0.2330,21.4053,1.0000,strong_candidate,Diagnostic only; final variable selection happ...
2,2007,productivity_gdp_per_hour,productivity_level,Productivity level volatility,40,40,8.0000,8,8,2.7752,0.3041,5.4931,1.0000,strong_candidate,Diagnostic only; final variable selection happ...
3,2007,unemployment_rate,unemployment,Unemployment volatility,51,48,8.0000,1,8,0.8717,0.2387,4.7675,0.9412,strong_candidate,Diagnostic only; final variable selection happ...
4,2019,gdp_growth,gdp_growth,GDP growth volatility,44,44,8.0000,8,8,1.2124,0.3842,7.9289,1.0000,strong_candidate,Diagnostic only; final variable selection happ...
5,2019,inflation_cpi,inflation,Inflation volatility,48,47,8.0000,2,8,1.0066,0.4288,3.9773,0.9792,strong_candidate,Diagnostic only; final variable selection happ...
6,2019,productivity_gdp_per_hour,productivity_level,Productivity level volatility,41,41,8.0000,8,8,1.6623,0.2607,12.3801,1.0000,strong_candidate,Diagnostic only; final variable selection happ...
7,2019,unemployment_rate,unemployment,Unemployment volatility,52,51,8.0000,3,8,1.1169,0.1771,4.6100,0.9808,strong_candidate,Diagnostic only; final variable selection happ...


In [12]:
# ------------------------------------------------------
# OUTPUT DATA DICTIONARY
# ------------------------------------------------------

data_dictionary = pd.DataFrame([
    {
        "file": "volatility_features_country_shock_variable_long.csv",
        "column": "volatility_std",
        "description": "Sample standard deviation of the variable in the pre-shock window. Lower generally means more stable.",
    },
    {
        "file": "volatility_features_country_shock_variable_long.csv",
        "column": "volatility_mad",
        "description": "Mean absolute deviation around the mean in the pre-shock window. Lower generally means more stable.",
    },
    {
        "file": "volatility_features_country_shock_variable_long.csv",
        "column": "stability_negative_std",
        "description": "Direction-aligned stability measure equal to negative volatility_std. Higher = better / more stable.",
    },
    {
        "file": "volatility_features_country_shock_variable_long.csv",
        "column": "stability_std_0_100",
        "description": "Within-shock-variable 0-100 stability score derived from volatility_std. Higher = better. Useful for diagnostics/visuals.",
    },
    {
        "file": "gdp_growth_stability_features_country_shock.csv",
        "column": "stability_negative_std",
        "description": "Preferred raw direction-aligned GDP growth stability candidate. Higher = better because it is negative GDP growth volatility.",
    },
    {
        "file": "gdp_growth_stability_features_country_shock.csv",
        "column": "stability_std_0_100",
        "description": "Scaled GDP growth stability score within shock window. Higher = better.",
    },
])

data_dictionary.to_csv(
    PROCESSED_DIR / "volatility_features_data_dictionary.csv",
    index=False,
)

data_dictionary.to_csv(
    DIAGNOSTICS_DIR / "volatility_features_data_dictionary.csv",
    index=False,
)

methodological_notes = pd.DataFrame([
    {
        "topic": "Pre-shock only",
        "note": "Volatility features use only pre-shock years: 2000-2007 for the 2007 shock and 2012-2019 for the 2019 shock.",
    },
    {
        "topic": "No leakage",
        "note": "Recovery outcomes are not used to construct the volatility/stability features.",
    },
    {
        "topic": "Direction alignment",
        "note": "Volatility is lower-is-better, so stability features are created as higher-is-better versions.",
    },
    {
        "topic": "Preferred feature",
        "note": "GDP growth stability is the preferred first volatility-derived candidate. Other volatility variables should be treated as candidates and checked for redundancy.",
    },
    {
        "topic": "No final selection",
        "note": "This notebook does not decide the final POSet ordering variables.",
    },
])

methodological_notes.to_csv(
    DIAGNOSTICS_DIR / "volatility_methodological_notes.csv",
    index=False,
)

display(data_dictionary)
display(methodological_notes)

,file,column,description
0,volatility_features_country_shock_variable_lon...,volatility_std,Sample standard deviation of the variable in t...
1,volatility_features_country_shock_variable_lon...,volatility_mad,Mean absolute deviation around the mean in the...
2,volatility_features_country_shock_variable_lon...,stability_negative_std,Direction-aligned stability measure equal to n...
3,volatility_features_country_shock_variable_lon...,stability_std_0_100,Within-shock-variable 0-100 stability score de...
4,gdp_growth_stability_features_country_shock.csv,stability_negative_std,Preferred raw direction-aligned GDP growth sta...
5,gdp_growth_stability_features_country_shock.csv,stability_std_0_100,Scaled GDP growth stability score within shock...


,topic,note
0,Pre-shock only,Volatility features use only pre-shock years: ...
1,No leakage,Recovery outcomes are not used to construct th...
2,Direction alignment,"Volatility is lower-is-better, so stability fe..."
3,Preferred feature,GDP growth stability is the preferred first vo...
4,No final selection,This notebook does not decide the final POSet ...


In [13]:
# ------------------------------------------------------
# CREATE VOLATILITY AUDIT WORKBOOK
# ------------------------------------------------------

VOLATILITY_AUDIT_XLSX = AUDIT_DIR / "04_Volatility_Features_Audit.xlsx"

audit_sources = [
    {
        "sheet_name": "01_window_coverage",
        "description": "Coverage summary by volatility variable and pre-shock window.",
        "path": DIAGNOSTICS_DIR / "volatility_window_coverage_summary.csv",
    },
    {
        "sheet_name": "02_long_features",
        "description": "Long volatility/stability feature table by country, shock, and variable.",
        "path": PROCESSED_DIR / "volatility_features_country_shock_variable_long.csv",
    },
    {
        "sheet_name": "03_country_shock_wide",
        "description": "Wide volatility/stability feature table by country-shock.",
        "path": PROCESSED_DIR / "volatility_features_country_shock_wide.csv",
    },
    {
        "sheet_name": "04_country_wide",
        "description": "One-row-per-country volatility/stability feature table across both shocks.",
        "path": PROCESSED_DIR / "volatility_features_country_wide.csv",
    },
    {
        "sheet_name": "05_gdp_stability",
        "description": "Preferred GDP growth stability feature output.",
        "path": PROCESSED_DIR / "gdp_growth_stability_features_country_shock.csv",
    },
    {
        "sheet_name": "06_quality_summary",
        "description": "Feature quality and coverage recommendation summary.",
        "path": DIAGNOSTICS_DIR / "volatility_feature_quality_summary.csv",
    },
    {
        "sheet_name": "07_recovery_diag",
        "description": "Diagnostic correlation with recovery outcomes; not causal and not used in feature construction.",
        "path": DIAGNOSTICS_DIR / "volatility_recovery_diagnostic_correlations.csv",
    },
    {
        "sheet_name": "08_dictionary",
        "description": "Volatility output data dictionary.",
        "path": PROCESSED_DIR / "volatility_features_data_dictionary.csv",
    },
    {
        "sheet_name": "09_method_notes",
        "description": "Methodological notes.",
        "path": DIAGNOSTICS_DIR / "volatility_methodological_notes.csv",
    },
]


def safe_sheet_name(name, used):
    cleaned = re.sub(r"[/\\*\[\]:?]", "_", str(name))[:31]
    base = cleaned
    i = 1

    while cleaned in used:
        suffix = f"_{i}"
        cleaned = base[:31 - len(suffix)] + suffix
        i += 1

    used.add(cleaned)
    return cleaned


def estimate_width(series, col_name, min_width=10, max_width=60):
    values = [str(col_name)] + series.dropna().astype(str).head(200).tolist()

    if not values:
        return min_width

    return max(min_width, min(max(len(v) for v in values) + 2, max_width))


used_sheets = set()

with pd.ExcelWriter(VOLATILITY_AUDIT_XLSX, engine="xlsxwriter") as writer:
    workbook = writer.book

    title_fmt = workbook.add_format({
        "bold": True,
        "font_size": 15,
        "font_color": "white",
        "bg_color": "#1F4E78",
        "align": "left",
        "valign": "vcenter",
    })

    desc_fmt = workbook.add_format({
        "text_wrap": True,
        "font_size": 10,
        "font_color": "#333333",
        "valign": "top",
    })

    header_fmt = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#5B9BD5",
        "border": 1,
        "align": "center",
        "valign": "vcenter",
        "text_wrap": True,
    })

    index_rows = []

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_temp = pd.read_csv(path)
                rows = len(df_temp)
                cols = len(df_temp.columns)
            except Exception:
                rows = np.nan
                cols = np.nan
        else:
            rows = 0
            cols = 0

        index_rows.append({
            "Sheet": item["sheet_name"],
            "Rows": rows,
            "Columns": cols,
            "Description": item["description"],
            "Path": str(path),
        })

    index_df = pd.DataFrame(index_rows)
    index_df.to_excel(writer, sheet_name="00_INDEX", index=False, startrow=5)

    ws = writer.sheets["00_INDEX"]
    ws.merge_range(0, 0, 0, max(4, len(index_df.columns) - 1), "04 Volatility Features Audit", title_fmt)
    ws.merge_range(1, 0, 3, max(4, len(index_df.columns) - 1), "Audit workbook for pre-shock volatility and stability feature construction.", desc_fmt)

    for col_idx, col_name in enumerate(index_df.columns):
        ws.write(5, col_idx, col_name, header_fmt)
        ws.set_column(col_idx, col_idx, estimate_width(index_df[col_name], col_name))

    ws.autofilter(5, 0, 5 + len(index_df), len(index_df.columns) - 1)
    ws.freeze_panes(6, 0)

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_sheet = pd.read_csv(path)
            except Exception as e:
                df_sheet = pd.DataFrame({"message": [f"Could not read file: {e}"]})
        else:
            df_sheet = pd.DataFrame({"message": ["File not found."]})

        if df_sheet.empty:
            df_sheet = pd.DataFrame({"message": ["No rows in this table."]})

        sheet_name = safe_sheet_name(item["sheet_name"], used_sheets)

        df_sheet.to_excel(writer, sheet_name=sheet_name, index=False, startrow=4)

        ws = writer.sheets[sheet_name]
        max_col = max(4, len(df_sheet.columns) - 1)

        ws.merge_range(0, 0, 0, max_col, item["sheet_name"], title_fmt)
        ws.merge_range(1, 0, 3, max_col, item["description"], desc_fmt)

        for col_idx, col_name in enumerate(df_sheet.columns):
            ws.write(4, col_idx, col_name, header_fmt)
            ws.set_column(col_idx, col_idx, estimate_width(df_sheet[col_name], col_name))

        ws.autofilter(4, 0, 4 + len(df_sheet), len(df_sheet.columns) - 1)
        ws.freeze_panes(5, 0)

print("Volatility audit workbook created:")
print(VOLATILITY_AUDIT_XLSX.resolve())

Volatility audit workbook created:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\04_Volatility_Features_Audit.xlsx


In [14]:
# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

print("04 VOLATILITY FEATURES COMPLETE")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE.resolve())

print("\nProcessed folder:")
print(PROCESSED_DIR.resolve())

print("\nDiagnostics folder:")
print(DIAGNOSTICS_DIR.resolve())

print("\nAudit workbook:")
print(VOLATILITY_AUDIT_XLSX.resolve())

print("\nMain processed outputs:")
main_outputs = [
    "volatility_features_country_shock_variable_long.csv",
    "volatility_features_country_shock_wide.csv",
    "volatility_features_country_wide.csv",
    "gdp_growth_stability_features_country_shock.csv",
    "volatility_features_data_dictionary.csv",
]

for file_name in main_outputs:
    path = PROCESSED_DIR / file_name
    status = "FOUND" if path.exists() else "MISSING"
    print(f"- {status}: {file_name}")

print("\nFeature quality summary:")
display(feature_quality_summary)

print("\nImportant notes:")
print("1. Only pre-shock years were used.")
print("2. Recovery outcomes were not used to construct features.")
print("3. Volatility is lower-is-better; stability versions are higher-is-better.")
print("4. GDP growth stability is the preferred first candidate from this notebook.")
print("5. Final variable selection still happens later in Pre-POSet EDA.")

print("\nNext notebook:")
print("05_Master_Dataset_Build.ipynb")

04 VOLATILITY FEATURES COMPLETE

Input file:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\00_Comparable_Raw_Files\Combined_Long\all_comparable_sources_long.csv

Processed folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Volatility_Features

Diagnostics folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\04_Volatility_Features

Audit workbook:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\04_Volatility_Features_Audit.xlsx

Main processed outputs:
- FOUND: volatility_features_country_shock_variable_long.csv
- FOUND: volatility_features_country_shock_wide.csv
- FOUND: volatility_features_country_wide.csv
- FOUND: gdp_growth_stability_features_country_shock.csv


,shock_id,variable,feature_slug,concept,countries_with_any_data,countries_with_sufficient_years,median_years_available,min_years_available,max_years_available,median_volatility_std,min_volatility_std,max_volatility_std,sufficient_years_country_rate,feature_quality_recommendation,note
0,2007,gdp_growth,gdp_growth,GDP growth volatility,44,44,8.0000,8,8,1.5573,0.7130,4.7569,1.0000,strong_candidate,Diagnostic only; final variable selection happ...
1,2007,inflation_cpi,inflation,Inflation volatility,47,47,8.0000,8,8,1.0300,0.2330,21.4053,1.0000,strong_candidate,Diagnostic only; final variable selection happ...
2,2007,productivity_gdp_per_hour,productivity_level,Productivity level volatility,40,40,8.0000,8,8,2.7752,0.3041,5.4931,1.0000,strong_candidate,Diagnostic only; final variable selection happ...
3,2007,unemployment_rate,unemployment,Unemployment volatility,51,48,8.0000,1,8,0.8717,0.2387,4.7675,0.9412,strong_candidate,Diagnostic only; final variable selection happ...
4,2019,gdp_growth,gdp_growth,GDP growth volatility,44,44,8.0000,8,8,1.2124,0.3842,7.9289,1.0000,strong_candidate,Diagnostic only; final variable selection happ...
5,2019,inflation_cpi,inflation,Inflation volatility,48,47,8.0000,2,8,1.0066,0.4288,3.9773,0.9792,strong_candidate,Diagnostic only; final variable selection happ...
6,2019,productivity_gdp_per_hour,productivity_level,Productivity level volatility,41,41,8.0000,8,8,1.6623,0.2607,12.3801,1.0000,strong_candidate,Diagnostic only; final variable selection happ...
7,2019,unemployment_rate,unemployment,Unemployment volatility,52,51,8.0000,3,8,1.1169,0.1771,4.6100,0.9808,strong_candidate,Diagnostic only; final variable selection happ...



Important notes:
1. Only pre-shock years were used.
2. Recovery outcomes were not used to construct features.
3. Volatility is lower-is-better; stability versions are higher-is-better.
4. GDP growth stability is the preferred first candidate from this notebook.
5. Final variable selection still happens later in Pre-POSet EDA.

Next notebook:
05_Master_Dataset_Build.ipynb
